## Create the Feature Detector Module

## Create the Feature Detector Module

### Introduction and Quick Recall

Welcome back! In the previous unit, we explored Harris corners and used them as a visual diagnostic for image texture. While that was a great starting point, real-world panoramas require algorithms that are stronger and more reliable when handling scale, rotation, and complex textured scenes.

In this lesson, we will upgrade our toolkit. We are going to build a clean, reusable feature detection module named `features.py`.

By storing our detection logic in a single reusable file, we will not have to rewrite it every time we want to inspect or match image features. Think of it as organizing tools into a toolbox: whenever you need a specific detector, it is ready and waiting.

As a quick reminder, in our CodeSignal environment, powerful libraries like OpenCV (`cv2`) are already pre-installed for you. You will not need to worry about setting up your environment or installing libraries — we can dive straight into writing code!

---

### Comparing SIFT, ORB, and AKAZE

Each feature method has unique strengths and produces a different type of descriptor, which is the numeric "fingerprint" around a keypoint.

| Algorithm | Full Name | Primary Strength | Descriptor Type |
| --- | --- | --- | --- |
| **SIFT** | Scale-Invariant Feature Transform | Highly accurate and robust to scale/rotation. | 128-dimensional floating point vectors. |
| **ORB** | Oriented FAST and Rotated BRIEF | Extremely fast and efficient; ideal for real-time apps. | 32-byte binary strings (integers). |
| **AKAZE** | Accelerated-KAZE | A balanced choice that preserves image edges. | Binary strings (integers). |

Here is how to interpret the table:

* SIFT is the heavy lifter. It is usually the strongest default for panorama-style matching because it handles scale and rotation well.
* ORB is the speedster. It uses compact binary descriptors, so it is fast and memory efficient.
* AKAZE is a modern middle ground. It often performs well on textured scenes with strong edges.

Separating this comparison from the code keeps the lesson focused: first we choose the tool conceptually, then we implement a factory that creates it.

---

### Building the Detector Factory

Our first reusable function is a detector factory. It returns the appropriate OpenCV detector based on the `method` argument.

```python
import cv2

def create_detector(method="sift", nfeatures=2000):
    if method == "sift":
        if not hasattr(cv2, "SIFT_create"):
            raise ValueError("SIFT is not available in this OpenCV build")
        return cv2.SIFT_create(nfeatures=nfeatures)
    if method == "orb":
        return cv2.ORB_create(nfeatures=nfeatures)
    if method == "akaze":
        return cv2.AKAZE_create()
    raise ValueError(f"Unknown feature method: {method}")

```

The SIFT branch includes an availability check because older OpenCV builds sometimes placed SIFT in an extra module or did not include it at all. Modern OpenCV builds usually include it, but the explicit check gives learners a clear error message if it is missing.

We do not add the same availability check for ORB and AKAZE here because they are standard OpenCV feature detectors in the environment used for this course. If a very unusual OpenCV build were missing one of them, the detector creation call would fail immediately. For our course pipeline, the SIFT check is the most helpful one because it is the detector with the most common version-related history.

Notice that `AKAZE_create()` does not receive `nfeatures`. OpenCV's AKAZE constructor does not use the same simple maximum-feature argument that SIFT and ORB do, so we call it without that option.

---

### Detecting Keypoints and Computing Descriptors

Now that we can create a detector, we need one clean entry point for feature extraction. This helper accepts a grayscale image, creates the requested detector, and returns keypoints plus descriptors.

```python
def detect_and_compute(gray, method="sift", nfeatures=2000):
    detector = create_detector(method=method, nfeatures=nfeatures)
    keypoints, descriptors = detector.detectAndCompute(gray, None)
    return keypoints, descriptors

```

The returned values are:

* **Keypoints:** locations, sizes, and orientations of interesting image points.
* **Descriptors:** numeric fingerprints describing the local appearance around each keypoint.

We pass `None` as the second argument because we are not masking the image. The detector is allowed to search everywhere.

---

### Integrating and Visualizing in the Main Script

With our `features.py` module ready, let's switch to our main program, `solution.py`, to test it.

First, we will set up the script to accept command-line inputs using `argparse`. We will also import our new `detect_and_compute` function alongside the helpful image reading tools we built in previous lessons (located in our `cvkit` module).

```python
import argparse
import cv2
from cvkit import make_panel, preprocess_for_features, read_color
from features import detect_and_compute

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("path")
    parser.add_argument("--method", choices=["sift", "orb", "akaze"], default="sift")
    parser.add_argument("--nfeatures", type=int, default=2000)
    parser.add_argument("--preview-size", type=int, default=900)
    args = parser.parse_args()

```

Next, let's load our image, convert it to grayscale using our pre-built preprocessing tool, and pass it to our new feature detector.

```python
    image = read_color(args.path)
    gray = preprocess_for_features(image)
    keypoints, descriptors = detect_and_compute(
        gray,
        method=args.method,
        nfeatures=args.nfeatures,
    )

```

To visualize what the computer found, we can use OpenCV's `drawKeypoints` function. We will use a special flag called `DRAW_RICH_KEYPOINTS`. This draws circles around the features to indicate their size and orientation.

```python
    output = cv2.drawKeypoints(
        image,
        keypoints,
        None,
        flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
    )

```

Printing the shape and data type of our descriptors is beneficial for grounding these concepts in real data. This helps us understand what will be passed to our matching algorithms later. Let's print the details and then display the image on the screen.

```python
    print("method:", args.method)
    print("keypoints:", len(keypoints))
    print("descriptor shape:", None if descriptors is None else descriptors.shape)
    print("descriptor dtype:", None if descriptors is None else descriptors.dtype)
    cv2.imshow(
        "keypoints",
        make_panel(
            [
                ("input", gray),
                ("keypoints", output),
            ],
            max_size=args.preview_size,
        ),
    )
    cv2.waitKey(0)
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

```

When you run this code on an image, the terminal output will look similar to this:

```text
method: sift
keypoints: 2000
descriptor shape: (2000, 128)
descriptor dtype: float32

```

This tells you that SIFT found the maximum number of requested keypoints, produced one descriptor row per keypoint, and used 128 floating-point values per descriptor.

Try comparing methods from the terminal:

```text
python solution.py sample_image.jpg --method sift
python solution.py sample_image.jpg --method orb
python solution.py sample_image.jpg --method akaze

```

Look for differences in keypoint count, descriptor shape, descriptor type, and the visual density of keypoints in the preview.

---

### Summary and Next Steps

Great job! You have successfully built a standardized, reusable module for extracting features from images using powerful algorithms like SIFT, ORB, and AKAZE. By cleanly separating this logic into `features.py`, we have made our main script much easier to read and maintain.

This step perfectly prepares us for the next phase of our image stitching journey. Now that we have reliable feature fingerprints, we can learn how to match these fingerprints between two separate images to determine how they overlap.

You are now ready to proceed to the interactive practices! These exercises will provide hands-on experience in writing and testing these detection functions yourself. Good luck, and have fun!

## Wiring Up the Command Line

Kick off the image stitching pipeline by providing your script with a proper command-line interface so that subsequent steps can incorporate the feature detection module you will build next.

Open solution.py and fill in the main function step by step:

    Create an ArgumentParser and store it in parser.
    Add a positional path argument for the input image.
    Add a --method option that accepts only "sift", "orb", or "akaze", defaulting to "sift".
    Add an --nfeatures option of type int with a default of 2000.
    Add a --preview-size option of type int with a default of 900.
    Call parse_args() on the parser and store the result in args.

Once these flags are configured, every subsequent component of the pipeline (detector choice, feature count, and preview window) can be controlled directly from the terminal. Setting up a clean CLI now will save you significant friction later.

```
import argparse


def main():
    # TODO: Create an ArgumentParser instance and store it in a variable called `parser`.

    # TODO: Add a positional argument called `path`.

    # TODO: Add a `--method` argument with choices ["sift", "orb", "akaze"] and default "sift".

    # TODO: Add a `--nfeatures` argument of type int with default 2000.

    # TODO: Add a `--preview-size` argument of type int with default 900.

    # TODO: Call `parse_args()` on the parser and store the result in a variable called `args`.
    pass


if __name__ == "__main__":
    main()

```

```python
import argparse


def main():
    # Create an ArgumentParser instance and store it in a variable called `parser`.
    parser = argparse.ArgumentParser()

    # Add a positional argument called `path`.
    parser.add_argument("path")

    # Add a `--method` argument with choices ["sift", "orb", "akaze"] and default "sift".
    parser.add_argument("--method", choices=["sift", "orb", "akaze"], default="sift")

    # Add a `--nfeatures` argument of type int with default 2000.
    parser.add_argument("--nfeatures", type=int, default=2000)

    # Add a `--preview-size` argument of type int with default 900.
    parser.add_argument("--preview-size", type=int, default=900)

    # Call `parse_args()` on the parser and store the result in a variable called `args`.
    args = parser.parse_args()


if __name__ == "__main__":
    main()

``` 

## Building the Feature Detector Factory

With the command-line "control panel" ready, it is time to give the pipeline a brain that can pick a feature detector on demand.

Open features.py and complete the create_detector factory function. It should return the right OpenCV detector based on the method argument:

    For "sift": First, check that cv2 has the SIFT_create attribute. If it does not, raise a ValueError stating that SIFT is not available. Otherwise, return cv2.SIFT_create(nfeatures=nfeatures).
    For "orb": Return cv2.ORB_create(nfeatures=nfeatures).
    For "akaze": Return cv2.AKAZE_create() (this one does not take nfeatures).
    For anything else: Raise a ValueError with the message f"Unknown feature method: {method}".

We explicitly check SIFT because SIFT has the most common OpenCV version-history issue. ORB and AKAZE are standard in the course environment, so creating them directly keeps the factory simple; if an unusual OpenCV build lacked one, the creation call itself would fail clearly.

This factory is the piece that lets the rest of the pipeline stay clean and swap algorithms with a single flag, so it will pay off many times over.


```
import cv2


def create_detector(method="sift", nfeatures=2000):
    # TODO: If method is "sift", first check that `cv2` has the attribute "SIFT_create".
    #       If not, raise a ValueError explaining that SIFT is unavailable.
    #       Otherwise, return `cv2.SIFT_create(nfeatures=nfeatures)`.

    # TODO: If method is "orb", return `cv2.ORB_create(nfeatures=nfeatures)`.

    # TODO: If method is "akaze", return `cv2.AKAZE_create()` (no nfeatures argument).

    # TODO: If none of the methods above matched, raise a ValueError
    #       with the message f"Unknown feature method: {method}".
    pass

```

```python
import cv2


def create_detector(method="sift", nfeatures=2000):
    # If method is "sift", first check that `cv2` has the attribute "SIFT_create".
    # If not, raise a ValueError explaining that SIFT is unavailable.
    # Otherwise, return `cv2.SIFT_create(nfeatures=nfeatures)`.
    if method == "sift":
        if not hasattr(cv2, "SIFT_create"):
            raise ValueError("SIFT is not available in this OpenCV build")
        return cv2.SIFT_create(nfeatures=nfeatures)

    # If method is "orb", return `cv2.ORB_create(nfeatures=nfeatures)`.
    if method == "orb":
        return cv2.ORB_create(nfeatures=nfeatures)

    # If method is "akaze", return `cv2.AKAZE_create()` (no nfeatures argument).
    # Some OpenCV builds only expose AKAZE through the class-based factory
    # (`cv2.AKAZE.create()`) instead of the top-level `cv2.AKAZE_create()`
    # function, so fall back to that form if needed.
    if method == "akaze":
        if hasattr(cv2, "AKAZE_create"):
            return cv2.AKAZE_create()
        if hasattr(cv2, "AKAZE"):
            return cv2.AKAZE.create()
        raise ValueError("AKAZE is not available in this OpenCV build")

    # If none of the methods above matched, raise a ValueError
    # with the message f"Unknown feature method: {method}".
    raise ValueError(f"Unknown feature method: {method}")

```

## Detecting Keypoints and Computing Descriptors

With create_detector ready to hand you any detector you ask for, it's time to actually put it to work.

In this exercise, you'll finish the detect_and_compute function in features.py. This helper is the one your pipeline will call whenever it needs keypoints and descriptors from a grayscale image.

Follow the TODOs inside the function:

    Use create_detector with the method and nfeatures arguments to build a detector.
    Call its detectAndCompute(gray, None) method to get the keypoints and descriptors.
    Return both as a tuple in the order (keypoints, descriptors).

Once this is done, your pipeline will have a single, clean entry point for feature extraction — no matter which algorithm is chosen.

```
Python
import cv2


def create_detector(method="sift", nfeatures=2000):
    if method == "sift":
        if not hasattr(cv2, "SIFT_create"):
            raise ValueError("SIFT is not available in this OpenCV build")
        return cv2.SIFT_create(nfeatures=nfeatures)
    
    if method == "orb":
        return cv2.ORB_create(nfeatures=nfeatures)
    
    if method == "akaze":
        if hasattr(cv2, "AKAZE_create"):
            return cv2.AKAZE_create()
        if hasattr(cv2, "AKAZE"):
            return cv2.AKAZE.create()
        raise ValueError("AKAZE is not available in this OpenCV build")

    raise ValueError(f"Unknown feature method: {method}")

```

Here is the complete code for `features.py` containing both `create_detector` and the completed `detect_and_compute` function:

```python
import cv2


def create_detector(method="sift", nfeatures=2000):
    if method == "sift":
        if not hasattr(cv2, "SIFT_create"):
            raise ValueError("SIFT is not available in this OpenCV build")
        return cv2.SIFT_create(nfeatures=nfeatures)
    
    if method == "orb":
        return cv2.ORB_create(nfeatures=nfeatures)
    
    if method == "akaze":
        if hasattr(cv2, "AKAZE_create"):
            return cv2.AKAZE_create()
        if hasattr(cv2, "AKAZE"):
            return cv2.AKAZE.create()
        raise ValueError("AKAZE is not available in this OpenCV build")

    raise ValueError(f"Unknown feature method: {method}")


def detect_and_compute(gray, method="sift", nfeatures=2000):
    # Use create_detector with the method and nfeatures arguments to build a detector.
    detector = create_detector(method=method, nfeatures=nfeatures)
    
    # Call its detectAndCompute(gray, None) method to get the keypoints and descriptors.
    keypoints, descriptors = detector.detectAndCompute(gray, None)
    
    # Return both as a tuple in the order (keypoints, descriptors).
    return keypoints, descriptors

```

## Bringing the Detection Pipeline to Life

With features.py ready to go, it's time to bring the whole pipeline together inside main() and finally see those keypoints on screen.

Your job is to fill in the body of main() step by step using the helpers from cvkit and features. Each TODO comment describes one small piece of the flow, from reading the image all the way to closing the preview window.

Work through them in order:

    Load the image with read_color and convert it to grayscale with preprocess_for_features.
    Call detect_and_compute, forwarding args.method and args.nfeatures, and unpack the result.
    Draw the keypoints onto the original image with cv2.drawKeypoints using the DRAW_RICH_KEYPOINTS flag.
    Print the four info lines (method, keypoint count, descriptor shape, descriptor dtype).
    Show the panel built by make_panel under the window title "keypoints", then wait for a key and destroy the windows.

After running the script, check the terminal before closing the preview window. Confirm that the printed method matches your flag, the keypoint count is reasonable, and the descriptor shape and dtype match the detector you selected. This is the moment all the previous pieces click into one working tool — take your time and enjoy the result.

```
import argparse
import cv2

from cvkit import make_panel, preprocess_for_features, read_color
from features import detect_and_compute


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("path")
    parser.add_argument("--method", choices=["sift", "orb", "akaze"], default="sift")
    parser.add_argument("--nfeatures", type=int, default=2000)
    parser.add_argument("--preview-size", type=int, default=900)
    args = parser.parse_args()

    # TODO: Read the input image with `read_color(args.path)` and store it in a variable called `image`.

    # TODO: Convert it to grayscale using `preprocess_for_features(image)` and store it in `gray`.

    # TODO: Call `detect_and_compute` on `gray`, forwarding `method=args.method`
    #       and `nfeatures=args.nfeatures`. Unpack the result into `keypoints, descriptors`.

    # TODO: Use `cv2.drawKeypoints` on `image` and `keypoints` (pass `None` as the third argument).
    #       Use the flag `cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS`. Store the result in `output`.

    # TODO: Print four lines:
    #         "method:" followed by args.method
    #         "keypoints:" followed by the number of detected keypoints
    #         "descriptor shape:" followed by descriptors.shape (or None if descriptors is None)
    #         "descriptor dtype:" followed by descriptors.dtype (or None if descriptors is None)

    # TODO: Display the result with `cv2.imshow("keypoints", ...)` where the second argument
    #       is `make_panel([("input", gray), ("keypoints", output)], max_size=args.preview_size)`.

    # TODO: Call `cv2.waitKey(0)` and then `cv2.destroyAllWindows()` to close the window cleanly.


if __name__ == "__main__":
    main()

```

```python
import argparse
import cv2

from cvkit import make_panel, preprocess_for_features, read_color
from features import detect_and_compute


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("path")
    parser.add_argument("--method", choices=["sift", "orb", "akaze"], default="sift")
    parser.add_argument("--nfeatures", type=int, default=2000)
    parser.add_argument("--preview-size", type=int, default=900)
    args = parser.parse_args()

    image = read_color(args.path)

    gray = preprocess_for_features(image)

    keypoints, descriptors = detect_and_compute(
        gray, method=args.method, nfeatures=args.nfeatures
    )

    output = cv2.drawKeypoints(
        image,
        keypoints,
        None,
        flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
    )

    print("method:", args.method)
    print("keypoints:", len(keypoints))
    print("descriptor shape:", descriptors.shape if descriptors is not None else None)
    print("descriptor dtype:", descriptors.dtype if descriptors is not None else None)

    cv2.imshow(
        "keypoints",
        make_panel([("input", gray), ("keypoints", output)], max_size=args.preview_size),
    )
    cv2.waitKey(0)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

```

## Timing the Three Detectors

Now that the detection pipeline is fully functional, it would be useful to know which method is the fastest on a given image — so let's add a small timer around detect_and_compute.

Start by importing the time module at the top of the file, alongside the other imports.

Inside main(), follow the TODOs to:

    Capture time.perf_counter() in start right before calling detect_and_compute.
    Compute elapsed = time.perf_counter() - start right after the call.
    Print an "elapsed (s):" line with round(elapsed, 4) next to the other information outputs.

With this in place, you will be able to compare SIFT, ORB, and AKAZE side by side. Run the script once per method and check the terminal output:
text

python solution.py sample_image.jpg --method sift
python solution.py sample_image.jpg --method orb
python solution.py sample_image.jpg --method akaze

Compare the "elapsed (s):" line, but also compare "keypoints:", "descriptor shape:", and "descriptor dtype:". Speed only matters if the detector still finds useful features.

```
import argparse
import cv2

from cvkit import make_panel, preprocess_for_features, read_color
from features import detect_and_compute

# TODO: Import the `time` module so we can measure how long detection takes.


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("path")
    parser.add_argument("--method", choices=["sift", "orb", "akaze"], default="sift")
    parser.add_argument("--nfeatures", type=int, default=2000)
    parser.add_argument("--preview-size", type=int, default=900)
    args = parser.parse_args()

    image = read_color(args.path)
    gray = preprocess_for_features(image)

    # TODO: Record the start time with `time.perf_counter()` and store it in `start`.

    keypoints, descriptors = detect_and_compute(
        gray,
        method=args.method,
        nfeatures=args.nfeatures,
    )

    # TODO: Compute the elapsed time as `time.perf_counter() - start` and store it in `elapsed`.

    output = cv2.drawKeypoints(
        image,
        keypoints,
        None,
        flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
    )

    print("method:", args.method)
    print("keypoints:", len(keypoints))
    print("descriptor shape:", None if descriptors is None else descriptors.shape)
    print("descriptor dtype:", None if descriptors is None else descriptors.dtype)
    # TODO: Print one more line: "elapsed (s):" followed by `round(elapsed, 4)`
    #       so the value is easy to read across runs.

    cv2.imshow(
        "keypoints",
        make_panel(
            [
                ("input", gray),
                ("keypoints", output),
            ],
            max_size=args.preview_size,
        ),
    )
    cv2.waitKey(0)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

```

Here's the full solution.py (with the AKAZE fix in features.py too, since it's needed for the akaze method to work in your test environment):

solution.py
```

import argparse
import cv2
import time

from cvkit import make_panel, preprocess_for_features, read_color
from features import detect_and_compute


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("path")
    parser.add_argument("--method", choices=["sift", "orb", "akaze"], default="sift")
    parser.add_argument("--nfeatures", type=int, default=2000)
    parser.add_argument("--preview-size", type=int, default=900)
    args = parser.parse_args()

    image = read_color(args.path)
    gray = preprocess_for_features(image)

    start = time.perf_counter()

    keypoints, descriptors = detect_and_compute(
        gray,
        method=args.method,
        nfeatures=args.nfeatures,
    )

    elapsed = time.perf_counter() - start

    output = cv2.drawKeypoints(
        image,
        keypoints,
        None,
        flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
    )

    print("method:", args.method)
    print("keypoints:", len(keypoints))
    print("descriptor shape:", None if descriptors is None else descriptors.shape)
    print("descriptor dtype:", None if descriptors is None else descriptors.dtype)
    print("elapsed (s):", round(elapsed, 4))

    cv2.imshow(
        "keypoints",
        make_panel(
            [
                ("input", gray),
                ("keypoints", output),
            ],
            max_size=args.preview_size,
        ),
    )
    cv2.waitKey(0)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()
```

features.py

```
import cv2


def create_detector(method="sift", nfeatures=2000):
    if method == "sift":
        if not hasattr(cv2, "SIFT_create"):
            raise ValueError("SIFT is not available in this OpenCV build")
        return cv2.SIFT_create(nfeatures=nfeatures)

    if method == "orb":
        return cv2.ORB_create(nfeatures=nfeatures)

    if method == "akaze":
        if hasattr(cv2, "AKAZE_create"):
            return cv2.AKAZE_create()
        if hasattr(cv2, "AKAZE"):
            return cv2.AKAZE.create()
        raise ValueError("AKAZE is not available in this OpenCV build")

    raise ValueError(f"Unknown feature method: {method}")


def detect_and_compute(gray, method="sift", nfeatures=2000):
    detector = create_detector(method=method, nfeatures=nfeatures)
    keypoints, descriptors = detector.detectAndCompute(gray, None)
    return keypoints, descriptors
```